Lab 4: LLMs and Prompt Engineering for Decision Support
Duration: 2 weeks [30 Jul - 13 Aug, 2026] Due Date: 13th August, 2026 Format: Jupyter Notebook / Google Colab + external APIs + GitHub version control Grading: This is a graded lab.

Student Name: Louisa-Lois Student ID: 13532028

Part 0: Repository and API-key setup

In [7]:
!git clone https://github.com/Louisa-Lois/lab-4-llm-decision-support.git
%cd lab-4-llm-decision-support

Cloning into 'lab-4-llm-decision-support'...
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 11 (delta 0), reused 11 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (11/11), 9.77 KiB | 9.77 MiB/s, done.
/content/lab-4-llm-decision-support/lab-4-llm-decision-support/lab-4-llm-decision-support


In [8]:
!pip install openai -q

In [9]:
# API-key setup

import os
from google.colab import userdata

API_KEY = userdata.get("GROQ_API_KEY")

# OpenAI-compatible client
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "openai/gpt-oss-120b"

print("Client ready.")

Client ready.


Section 1 — Talking to an LLM Programmatically

Part 1.1 — Your first API call

In [10]:
# Part 1.1: Your first API call

def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


# Call it once with a simple question and print the answer
answer = ask_llm("What is microfinance, in one sentence?")
print("Answer:")
print(answer)

Answer:
Microfinance is the provision of small‑scale financial services—such as loans, savings, and insurance—to low‑income individuals or groups who lack access to traditional banking.


In [11]:
# Print response.usage as well
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "What is microfinance, in one sentence?"},
    ],
    temperature=0.7,
    max_tokens=500,
)

print("\nToken usage:")
print(response.usage)


Token usage:
CompletionUsage(completion_tokens=100, prompt_tokens=89, total_tokens=189, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=54, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.089058426, prompt_time=0.0033799, completion_time=0.210357538, total_time=0.213737438)


**Student Reasoning - Anatomy of a call**

**1. Difference between system and user roles:**
The system role sets the model's overall behaviour, persona, and rules for the
entire conversation, it's set once and shapes how the model responds to
everything that follows. The user role is the actual question or task being
asked right now. Example: system = "You are a helpful assistant" (sets the
general behaviour), user = "What is microfinance, in one sentence?" (the
specific request). In this lab, the system role will later be used to give
the model a specific job (e.g. "You are an assistant to a microfinance loan
officer...") while the user role delivers the actual letter to process.

**2. What is a token, and why bill per token?**
A token is roughly a piece of a word, sometimes a whole word, sometimes
part of one. My test call used 89 prompt tokens (the question + system
message) and 76 completion tokens (the answer), totaling 165 tokens. API
providers bill per token rather than per request because the actual
computational cost of generating a response scales directly with how much
text is processed and produced, a one-word answer costs far less compute
than a 500-word essay, even though both are "one request." Billing per
token ties cost directly to actual resource usage.